In [2]:
import os
os.environ['SPARK_VERSION'] = '3.3'
os.environ["JAVA_HOME"] = '/usr/lib/jvm/java-8-openjdk-amd64/'

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Loan Data ETL Pipeline") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
spark.conf.set("spark.sql.execution.arrow.pyspark.fallback.enabled", "true")
spark.conf.set("spark.sql.execution.arrow.pyspark.memory.fraction", "0.2")
spark.conf.set("spark.sql.execution.arrow.pyspark.memory.max", "2g") 

25/06/16 15:52:34 WARN Utils: Your hostname, rohitkarki resolves to a loopback address: 127.0.1.1; using 10.13.164.166 instead (on interface wlp4s0)
25/06/16 15:52:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/rohitkarki/.local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/rohitkarki/.ivy2/cache
The jars for the packages stored in: /home/rohitkarki/.ivy2/jars
com.amazon.deequ#deequ added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-33ce81fa-db7b-441f-8a11-cdd2bb785193;1.0
	confs: [default]
	found com.amazon.deequ#deequ;2.0.11-spark-3.3 in central
	found org.scala-lang#scala-reflect;2.12.10 in central
	found org.scalanlp#breeze_2.12;1.2 in central
	found org.scalanlp#breeze-macros_2.12;1.2 in central
	found com.github.fommil.netlib#core;1.1.2 in central
	found net.sf.opencsv#opencsv;2.3 in central
	found com.github.wendykierp#JTransforms;3.1 in central
	found pl.edu.icm#JLargeArrays;1.5 in central
	found org.apache.commons#commons-math3;3.2 in central
	found com.chuusai#shapeless_2.12;2.3.3 in central
	found org.typelevel#macro-compat_2.12;1.1.1 in central
	found org.slf4j#slf4j-api;1.7.5 in central
	found org.typelevel#spire_2.12;0.17.0 in central
	found org.typelevel#spire-macros_2.12;

25/06/16 15:54:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [5]:
from pyspark.sql.functions import col, isnan, when, count, lit, split, to_date, hour, minute, second
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, TimestampType
from pyspark.sql.window import Window
from pyspark.sql import functions as F
# from pyspark.sql.functions import mode
# import pandas as pd
# import matplotlib.pyplot as plt
# import seaborn as sns
import os
from datetime import datetime


In [6]:
schema = StructType([
    StructField("Loan_id", StringType(), True),
    StructField("Gender", StringType(), True),
    StructField("Married", StringType(), True),
    StructField("Dependents", IntegerType(), True),
    StructField("Education", StringType(), True),
    StructField("Self_Employed", StringType(), True),
    StructField("ApplicantIncome", IntegerType(), True),
    StructField("CoapplicantIncome", IntegerType(), True),
    StructField("LoanAmount", IntegerType(), True),
    StructField("Loan_Amount_Term", IntegerType(), True),
    StructField("Credit_History", IntegerType(), True),
    StructField("Property_Area", StringType(), True),
    StructField("Loan_Status", StringType(), True),
])

In [7]:
try:
    input_path = "hdfs://localhost:9000/user/hive/warehouse/loan.csv"
    # Read the CSV file with the specified schema
    raw_df = spark.read.csv(input_path, header=True, schema=schema)

    print("CSV file read successfully.")
    print(f"Sample data")
    raw_df.show(5, truncate=False)

    # print("Initial data statistics:")
    # raw_df.describe().show()
    
    null_counts = raw_df.select([count(when(col(c).isNull() | isnan(col(c)), c)).alias(c) for c in raw_df.columns])
    null_counts.show(truncate=False)
    # null_counts.show()
except Exception as e:
    print(f"Error reading CSV file: {e}")

CSV file read successfully.
Sample data
+--------+------+-------+----------+------------+-------------+---------------+-----------------+----------+----------------+--------------+-------------+-----------+
|Loan_id |Gender|Married|Dependents|Education   |Self_Employed|ApplicantIncome|CoapplicantIncome|LoanAmount|Loan_Amount_Term|Credit_History|Property_Area|Loan_Status|
+--------+------+-------+----------+------------+-------------+---------------+-----------------+----------+----------------+--------------+-------------+-----------+
|LP001002|Male  |No     |0         |Graduate    |No           |5849           |0                |null      |360             |1             |Urban        |Y          |
|LP001003|Male  |Yes    |1         |Graduate    |No           |4583           |1508             |128       |360             |1             |Rural        |N          |
|LP001005|Male  |Yes    |0         |Graduate    |Yes          |3000           |0                |66        |360             |

In [7]:
raw_df.printSchema()

root
 |-- Loan_id: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Married: string (nullable = true)
 |-- Dependents: integer (nullable = true)
 |-- Education: string (nullable = true)
 |-- Self_Employed: string (nullable = true)
 |-- ApplicantIncome: integer (nullable = true)
 |-- CoapplicantIncome: integer (nullable = true)
 |-- LoanAmount: integer (nullable = true)
 |-- Loan_Amount_Term: integer (nullable = true)
 |-- Credit_History: integer (nullable = true)
 |-- Property_Area: string (nullable = true)
 |-- Loan_Status: string (nullable = true)



In [7]:
# Step 1: Replace NULL values with mode for each column
def replace_nulls_with_mode(df):
    transformed_df = df

    # Process each column
    for column in df.columns:
        # Skip date column as it will be processed separately
        if column == 'Application_Date':
            continue

        # Check if column has any null values
        null_count = df.filter(col(column).isNull() | isnan(col(column))).count()
        if null_count == 0:
            print(f"Column {column} has no NULL values. Skipping...")
            continue

        # For numerical columns (more complex mode calculation)
        if df.schema[column].dataType in [IntegerType(), DoubleType()]:
            # Calculate mode using the most frequent value
            mode_df = df.groupBy(column) \
                .count() \
                .filter(col(column).isNotNull() & (~isnan(col(column)))) \
                .orderBy("count", ascending=False)

            if mode_df.count() > 0:
                mode_val = mode_df.first()[column]
                transformed_df = transformed_df.withColumn(
                    column,
                    when(col(column).isNull() | isnan(col(column)), lit(mode_val)).otherwise(col(column))
                )
                print(f"Replaced NULL values in {column} with mode: {mode_val}")

        # For string columns
        elif df.schema[column].dataType == StringType():
            mode_df = df.groupBy(column) \
                .count() \
                .filter(col(column).isNotNull()) \
                .orderBy("count", ascending=False)
            print("mode df count is " + str(mode_df.count()))
            if mode_df.count() > 0:
                mode_val = mode_df.first()[column]
                transformed_df = transformed_df.withColumn(
                    column,
                    when(col(column).isNull(), lit(mode_val)).otherwise(col(column))
                )
                print(f"Replaced NULL values in {column} with mode: {mode_val}")

    return transformed_df

transformed_df = replace_nulls_with_mode(raw_df)

Column Loan_id has no NULL values. Skipping...
mode df count is 2
Replaced NULL values in Gender with mode: Male
mode df count is 2
Replaced NULL values in Married with mode: Yes
Replaced NULL values in Dependents with mode: 0
Column Education has no NULL values. Skipping...
mode df count is 2
Replaced NULL values in Self_Employed with mode: No
Column ApplicantIncome has no NULL values. Skipping...
Replaced NULL values in CoapplicantIncome with mode: 0
Replaced NULL values in LoanAmount with mode: 120
Replaced NULL values in Loan_Amount_Term with mode: 360
Replaced NULL values in Credit_History with mode: 1
Column Property_Area has no NULL values. Skipping...
Column Loan_Status has no NULL values. Skipping...


In [8]:
transformed_df.show(5, truncate=True)

+--------+------+-------+----------+------------+-------------+---------------+-----------------+----------+----------------+--------------+-------------+-----------+
| Loan_id|Gender|Married|Dependents|   Education|Self_Employed|ApplicantIncome|CoapplicantIncome|LoanAmount|Loan_Amount_Term|Credit_History|Property_Area|Loan_Status|
+--------+------+-------+----------+------------+-------------+---------------+-----------------+----------+----------------+--------------+-------------+-----------+
|LP001002|  Male|     No|         0|    Graduate|           No|           5849|                0|       120|             360|             1|        Urban|          Y|
|LP001003|  Male|    Yes|         1|    Graduate|           No|           4583|             1508|       128|             360|             1|        Rural|          N|
|LP001005|  Male|    Yes|         0|    Graduate|          Yes|           3000|                0|        66|             360|             1|        Urban|          Y

In [ ]:
# Step 2: Separate date field into date and time columns
print("\nSeparating date field into date and time components...")
transformed_df = transformed_df.withColumn(
    "loan_date_only",
    to_date(col("Application_Date"))
).withColumn(
    "loan_time_hour",
    hour(col("Application_Date"))
).drop("Application_Date")
transformed_df.show(5, truncate=True)

In [15]:
# Summary statistics for numerical columns
print("Computing summary statistics...")
numeric_cols = [f.name for f in transformed_df.schema.fields
                if isinstance(f.dataType, (IntegerType, DoubleType))]

# Exclude date/time columns from numeric stats
numeric_cols = [col for col in numeric_cols if not (col.startswith("Application_Time") or col == "Application_Date")]

summary_stats = transformed_df.select(numeric_cols).summary(
    "count", "min", "max", "mean", "stddev"
)
summary_stats.show(truncate=False)

Computing summary statistics...
+-------+------------------+-----------------+------------------+-----------------+-----------------+------------------+------------------+
|summary|Dependents        |ApplicantIncome  |CoapplicantIncome |LoanAmount       |Loan_Amount_Term |Credit_History    |Total_Income      |
+-------+------------------+-----------------+------------------+-----------------+-----------------+------------------+------------------+
|count  |614               |614              |614               |614              |614              |614               |614               |
|min    |0                 |150              |0                 |9                |12               |0                 |1442              |
|max    |2                 |81000            |41667             |700              |480              |1                 |81000             |
|mean   |0.495114006514658 |5403.459283387622|1619.614006514658 |145.4657980456026|342.4104234527687|0.8550488599348535|7023.073

In [17]:
# Loan status distribution
print("Computing loan status distribution...")
loan_status_dist = transformed_df.groupBy("Loan_Status").count().orderBy("count", ascending=False)
loan_status_dist.show(truncate=False)

# Gender distribution
gender_dist = transformed_df.groupBy("Gender").count().orderBy("count", ascending=False)
gender_dist.show(truncate=False)

Computing loan status distribution...
+-----------+-----+
|Loan_Status|count|
+-----------+-----+
|Y          |422  |
|N          |192  |
+-----------+-----+

+------+-----+
|Gender|count|
+------+-----+
|Male  |502  |
|Female|112  |
+------+-----+



In [ ]:
# Income vs Loan Amount analysis
print("Computing income vs loan amount analysis...")
transformed_df = transformed_df.withColumn(
    "Total_Income",
    col("ApplicantIncome") + col("CoapplicantIncome")
)

transformed_df = transformed_df.withColumn(
    "Income_Category",
    when(col("Total_Income") < 5000, "Low Income (< 5000)")
    .when(col("Total_Income").between(5000, 10000), "Middle Income (5000-10000)")
    .when(col("Total_Income").between(10001, 20000), "Upper Middle (10001-20000)")
    .otherwise("High Income (20000+)")
)

income_loan_analysis = transformed_df.groupBy("Income_Category") \
    .agg(
        F.count("Loan_id").alias("total_applications"),
        F.mean("LoanAmount").alias("avg_loan_amount"),
        F.sum(when(col("Loan_Status") == "Y", 1).otherwise(0)).alias("approved_count"),
        F.sum(when(col("Loan_Status") == "N", 1).otherwise(0)).alias("rejected_count")
    ) \
    .withColumn("approval_rate",
                F.round(col("approved_count") / (col("approved_count") + col("rejected_count")), 2)) \
    .orderBy("Income_Category")

# Credit history impact
credit_impact = transformed_df.groupBy("Credit_History") \
    .agg(
        F.count("Loan_id").alias("total_applications"),
        F.sum(when(col("Loan_Status") == "Y", 1).otherwise(0)).alias("approved_count")
    ) \
    .withColumn("approval_rate",
                F.round(col("approved_count") / col("total_applications"), 2)) \
    .orderBy("Credit_History")

# Property area analysis
property_analysis = transformed_df.groupBy("Property_Area") \
    .agg(
        F.count("Loan_id").alias("total_applications"),
        F.mean("LoanAmount").alias("avg_loan_amount"),
        F.sum(when(col("Loan_Status") == "Y", 1).otherwise(0)).alias("approved_count")
    ) \
    .withColumn("approval_rate",
                F.round(col("approved_count") / col("total_applications"), 2)) \
    .orderBy("Property_Area")

# Education and employment impact
education_employment = transformed_df.groupBy("Education", "Self_Employed") \
    .agg(
        F.count("Loan_id").alias("total_applications"),
        F.mean("LoanAmount").alias("avg_loan_amount"),
        F.sum(when(col("Loan_Status") == "Y", 1).otherwise(0)).alias("approved_count")
    ) \
    .withColumn("approval_rate",
                F.round(col("approved_count") / col("total_applications"), 2)) \
    .orderBy("Education", "Self_Employed")

# Store insights in separate DataFrames
def collect_insights():
    insights = {}
    insights["summary_stats"] = summary_stats.toPandas()
    insights["loan_status_dist"] = loan_status_dist.toPandas()
    insights["gender_dist"] = gender_dist.toPandas()
    insights["income_loan_analysis"] = income_loan_analysis.toPandas()
    insights["credit_impact"] = credit_impact.toPandas()
    insights["property_analysis"] = property_analysis.toPandas()
    insights["education_employment"] = education_employment.toPandas()

    return insights

insights = collect_insights()

# Print sample insights
print("\nSample Insights:")
print("\nLoan Status Distribution:")
print(insights["loan_status_dist"])

print("\nCredit History Impact on Approval Rate:")
print(insights["credit_impact"])

print("\nIncome Category Analysis:")
print(insights["income_loan_analysis"])

# LOAD: Write transformed data to output tables
print("\nLOAD PHASE: Writing data to output tables...")

# Output path for transformed data
output_base_path = "loan_data_output"

# Create output directory if it doesn't exist
os.makedirs(output_base_path, exist_ok=True)

# Save transformed data
transformed_df.write \
    .mode("overwrite") \
    .parquet(f"{output_base_path}/transformed_loan_data")
print(f"Transformed data saved to {output_base_path}/transformed_loan_data")

# Save insights to separate tables
os.makedirs(f"{output_base_path}/insights", exist_ok=True)
for insight_name, insight_df in insights.items():
    output_path = f"{output_base_path}/insights/{insight_name}"

    # Convert pandas dataframe back to spark dataframe
    spark_insight_df = spark.createDataFrame(insight_df)

    # Save as parquet
    spark_insight_df.write \
        .mode("overwrite") \
        .parquet(output_path)
    print(f"Insight '{insight_name}' saved to {output_path}")

# Generate visualization for key insights
print("\nGenerating visualizations...")

# Visualization directory
viz_dir = f"{output_base_path}/visualizations"
os.makedirs(viz_dir, exist_ok=True)

# 1. Loan Status Distribution
plt.figure(figsize=(10, 6))
sns.countplot(data=transformed_df.toPandas(), x="Loan_Status")
plt.title("Loan Status Distribution")
plt.savefig(f"{viz_dir}/loan_status_distribution.png")

# 2. Income vs Loan Amount
plt.figure(figsize=(10, 6))
income_data = transformed_df.select("ApplicantIncome", "LoanAmount", "Loan_Status").toPandas()
sns.scatterplot(data=income_data, x="ApplicantIncome", y="LoanAmount", hue="Loan_Status")
plt.title("Applicant Income vs Loan Amount")
plt.savefig(f"{viz_dir}/income_vs_loan_amount.png")

# 3. Credit History Impact
plt.figure(figsize=(10, 6))
credit_impact_data = insights["credit_impact"]
sns.barplot(data=credit_impact_data, x="Credit_History", y="approval_rate")
plt.title("Impact of Credit History on Loan Approval Rate")
plt.savefig(f"{viz_dir}/credit_history_impact.png")

# 4. Property Area Analysis
plt.figure(figsize=(10, 6))
property_data = insights["property_analysis"]
sns.barplot(data=property_data, x="Property_Area", y="approval_rate")
plt.title("Loan Approval Rate by Property Area")
plt.savefig(f"{viz_dir}/property_area_analysis.png")

# Generate consolidated report
report_path = f"{output_base_path}/loan_analysis_report.txt"
with open(report_path, "w") as f:
    f.write("LOAN DATA ANALYSIS REPORT\n")
    f.write("=========================\n\n")
    f.write(f"Report generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")

    f.write("1. DATASET OVERVIEW\n")
    f.write(f"   Total records processed: {transformed_df.count()}\n")
    f.write(f"   Approved loans: {loan_status_dist.filter(col('Loan_Status') == 'Y').select('count').collect()[0][0]}\n")
    f.write(f"   Rejected loans: {loan_status_dist.filter(col('Loan_Status') == 'N').select('count').collect()[0][0]}\n\n")

    f.write("2. KEY INSIGHTS\n")
    f.write("   a. Credit History Impact:\n")
    for row in insights["credit_impact"].itertuples():
        f.write(f"      - Credit History {row.Credit_History}: {row.approval_rate*100:.1f}% approval rate\n")

    f.write("\n   b. Property Area Analysis:\n")
    for row in insights["property_analysis"].itertuples():
        f.write(f"      - {row.Property_Area}: {row.approval_rate*100:.1f}% approval rate, Avg loan: {row.avg_loan_amount:.1f}\n")

    f.write("\n   c. Education and Employment:\n")
    for row in insights["education_employment"].itertuples():
        f.write(f"      - {row.Education}, Self-Employed={row.Self_Employed}: {row.approval_rate*100:.1f}% approval rate\n")

    f.write("\n3. RECOMMENDATIONS\n")
    f.write("   Based on the analysis, consider the following:\n")
    f.write("   - Credit history is a critical factor in loan approval\n")
    f.write("   - Income level correlates with loan amount and approval rate\n")
    f.write("   - Property area influences approval decisions\n")

print(f"Analysis report saved to {report_path}")

print("\nETL Pipeline completed successfully!")
print(f"All output saved to {output_base_path}")

# Show transformed data after all processing
print("\nSample of final transformed data:")
transformed_df.show(5, truncate=False)


In [16]:
import pyarrow as pa
import numpy as np

# Create a PyArrow Table
table = pa.table([pa.array(np.random.rand(100)) for i in range(3)], names=["a", "b", "c"])

# Create a Spark DataFrame from the PyArrow Table
df = spark.createDataFrame(table)

# Convert the Spark DataFrame to a PyArrow Table
# result_table = df.select("*").toArrow()

# print(result_table.schema)

TypeError: Can not infer schema for type: <class 'pyarrow.lib.ChunkedArray'>